# AI Lab - Practical Question No. 13
## BFS Shortest Path Constructor

- **Syllabus Area**: Uninformed Search Techniques
- **Difficulty**: 🟡 Intermediate
- **Practical Type**: Algorithm Design
- **Core AI Concept**: Path Reconstruction, Goal Testing (Early vs Late), FIFO Frontier Queue
- **Python Concepts Tested**: Path tracking, dictionary parent mapping, `collections.deque`
- **Expected Algorithm / Technique**: BFS with path tracking
- **Complexity Expectation**: Time Complexity: $O(V + E)$ | Space Complexity: $O(V)$
- **Constraints / Conditions**: Graph is unweighted.
- **Sample Input**: Start: A, Goal: F
- **Sample Output**: Shortest Path: `['A', 'C', 'F']`
- **University Practical Importance**: Very High
- **Optional Challenge**: Print all possible shortest paths if multiple exist.

---

### Problem Statement
Extend the standard BFS algorithm to not just traverse a graph, but to find and return the exact shortest path (in terms of number of edges) between a given source and destination vertex.
The program must reconstruct the path from the goal back to the start.

### Task Requirements
1. **Path Tracking**: Queue must store paths (lists of nodes) or maintain a parent dictionary.
2. **Immediate Termination**: Stop and return the path immediately when the goal is tested (Early Goal Testing).
3. **Robust Handling**: Handle cases where no path exists (unreachable goal or disconnected components).
4. **Exact Shortest Path**: Guarantee minimum edge count on unweighted graphs.
5. **Optional Challenge**: Discover and output all tied shortest paths if multiple minimal paths exist.

### 1. Theoretical Foundation & Algorithm Mechanics

#### 1. Path vs Traversal
- **Graph Traversal**: Explores every reachable node in a graph/component to collect or process vertices. Terminates only when the frontier is empty.
- **Shortest Path Constructor**: Finds a specific sequence of edges connecting a designated start node $v_0$ to a destination goal $v_k$ with minimal step cost.

#### 2. Early Goal Testing vs Late Goal Testing
- **Early Goal Testing (Generation Time)**: The goal test is applied immediately when a neighbor node is *generated* (before entering the queue). In unweighted graphs with uniform edge costs, the first time the goal is generated is guaranteed to be at minimal depth. This saves expanding all remaining nodes at that level, providing an asymptotic $b$-factor speedup!
- **Late Goal Testing (Expansion Time)**: Goal test is deferred until the node is popped from the queue. Necessary in non-uniform cost search (Dijkstra/$A^*$), but wasteful in standard BFS.

#### 3. Why BFS Finds the Shortest Path in Unweighted Graphs
- In unweighted graphs, every step cost $c = 1$. Path cost equals depth $d$.
- The FIFO queue guarantees monotonic non-decreasing exploration: all nodes at depth $d$ are expanded before any at depth $d + 1$.
- Inductive proof: No path of length $< d^*$ exists (or goal would have been found earlier), and no path of length $> d^*$ has been reached yet. Hence the first path to the goal is optimal.

#### 4. Parent Dictionary Backtracking ($O(V)$ Space)
- Instead of duplicating entire path lists in the queue ($O(V \cdot b^d)$ memory), we maintain `parents[child] = parent`.
- Upon reaching the goal, we walk backward from goal to start:
  $$\text{goal} \to \text{parent}[\text{goal}] \to \dots \to \text{start}$$
  Reversing this list yields the forward shortest path in $O(d)$ time and strictly $O(V)$ auxiliary space.

In [ ]:
# Setup workspace path and import BFS Shortest Path Constructor module
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.bfs_shortest_path import (
    DEFAULT_SHORTEST_PATH_GRAPH,
    CYCLIC_BENCHMARK_GRAPH,
    MULTI_PATH_DIAMOND_GRAPH,
    MULTI_PATH_GRID_GRAPH,
    DISCONNECTED_SHORTEST_PATH_GRAPH,
    bfs_shortest_path,
    find_all_shortest_paths,
    compare_goal_testing_strategies,
    compare_tracking_mechanisms,
    reconstruct_path_from_parents,
    visualize_shortest_path,
    get_viva_reference,
)
print("BFS Shortest Path Constructor module imported successfully!")

### 2. Default Sample Run (Start: A, Goal: F)
Let us run the BFS Shortest Path Constructor on the standard laboratory graph with Early Goal Testing.

In [ ]:
# Execute BFS Shortest Path Search on Sample Graph
result = bfs_shortest_path(DEFAULT_SHORTEST_PATH_GRAPH, start="A", goal="F")
print(result.format_summary())
print("\nParent Dictionary Mapping:")
print(result.format_parent_map())

### 3. Chronological Step-by-Step FIFO Frontier Trace
Observe how the early goal test triggers on Step 9 when neighbor `F` is generated from `C`, terminating the search immediately without expanding `D` or `E`!

In [ ]:
print(result.format_step_table())

### 4. Early Goal Testing vs Late Goal Testing Comparison
Compare early goal testing (evaluation on neighbor generation) vs late goal testing (evaluation upon queue expansion).

In [ ]:
comp = compare_goal_testing_strategies(DEFAULT_SHORTEST_PATH_GRAPH, start="A", goal="F")
print(f"Shortest Path Discovered: {comp['shortest_path']} (Cost: {comp['path_cost_edges']} edges)")
print(f"Early Testing Expansions : {comp['early_goal_test']['nodes_expanded']} nodes")
print(f"Late Testing Expansions  : {comp['late_goal_test']['nodes_expanded']} nodes")
print(f"Expansions Saved         : {comp['expansions_saved']} node(s) (50% reduction in expansions!)")

### 5. Tracking Mechanism Comparison: Parent Dict vs Queue-Stored Paths
Compare the memory-efficient Parent Dictionary Backtracking ($O(V)$ space) against Storing Full Path Lists in the FIFO queue.

In [ ]:
modes = compare_tracking_mechanisms(DEFAULT_SHORTEST_PATH_GRAPH, start="A", goal="F")
print(f"Identical Paths Discovered: {modes['identical_paths']} -> {modes['path']}")
print(f"Parent Dict Auxiliary Space: {modes['parent_dict']['space_complexity']}")
print(f"Queue Paths Auxiliary Space: {modes['queue_paths']['space_complexity']}")

### 6. Edge Cases: Cycles, Start == Goal, and Disconnected Components

In [ ]:
# 1. Cyclic Graph
cyclic_res = bfs_shortest_path(CYCLIC_BENCHMARK_GRAPH, start="A", goal="F")
print(f"Cyclic Graph Shortest Path: {cyclic_res.path} (Cost: {cyclic_res.cost_edges} edges)")

# 2. Start == Goal
identity_res = bfs_shortest_path(DEFAULT_SHORTEST_PATH_GRAPH, start="A", goal="A")
print(f"Identity Search (A -> A): {identity_res.path} (Cost: {identity_res.cost_edges} edges)")

# 3. Disconnected Graph (Unreachable Goal)
disconn_res = bfs_shortest_path(DISCONNECTED_SHORTEST_PATH_GRAPH, start="A", goal="F")
print(f"Disconnected Graph (A -> F): {disconn_res.path} (Goal Found: {disconn_res.goal_found})")

### 7. Optional Challenge: Finding All Possible Shortest Paths
When multiple distinct paths have identical minimum edge counts (e.g. Diamond Graph: A -> B -> D and A -> C -> D), our algorithm finds and enumerates all of them!

In [ ]:
all_diamond_paths = find_all_shortest_paths(MULTI_PATH_DIAMOND_GRAPH, start="A", goal="D")
print(f"All Shortest Paths in Diamond Graph ({len(all_diamond_paths)} found):")
for idx, p in enumerate(all_diamond_paths, start=1):
    print(f"  [{idx}] {' -> '.join(p)} (Length: {len(p) - 1} edges)")

all_grid_paths = find_all_shortest_paths(MULTI_PATH_GRID_GRAPH, start="A", goal="G")
print(f"\nAll Shortest Paths in Grid Graph ({len(all_grid_paths)} found):")
for idx, p in enumerate(all_grid_paths, start=1):
    print(f"  [{idx}] {' -> '.join(p)} (Length: {len(p) - 1} edges)")

### 8. Graphical Visualization (Matplotlib & NetworkX)
Visualizing the shortest path highlighted in coral/red with green start marker and red goal marker.

In [ ]:
import matplotlib.pyplot as plt
fig = visualize_shortest_path(result, show=False)
if fig:
    plt.show()

### 9. Comprehensive University Practical Viva Voce Reference Guide

#### Q1: What is the key difference between a Graph Traversal and a Shortest Path Constructor?
- **Traversal**: A general exploratory walk that visits all reachable vertices in an arbitrary or level-wise order without stopping at a target.
- **Path Constructor**: A targeted goal search returning the minimal sequence of valid transitions between two specific vertices.

#### Q2: What is the difference between Early Goal Testing and Late Goal Testing, and why does BFS use early testing?
- **Early Goal Testing**: Performed upon node *generation* (before enqueuing). Because all edges have uniform cost in unweighted graphs, the first generation of the goal is guaranteed to be minimal. This saves generating the next level of $b^{d+1}$ nodes and expands fewer vertices ($b$-factor reduction in time & memory).
- **Late Goal Testing**: Performed upon node *expansion* (after popping from queue). Essential for Dijkstra / $A^*$ where edge weights vary, but redundant in unweighted BFS.

#### Q3: Why is BFS guaranteed to find the shortest path in unweighted graphs?
- Unweighted edges imply uniform unit cost $c = 1$.
- The FIFO queue enforces a strictly non-decreasing search frontier: all paths of length $d$ are expanded before any path of length $d + 1$.
- By induction, the first path constructed to the goal must have minimal length.

#### Q4: Compare Parent Dictionary Backtracking with Queue-Stored Paths.
- **Parent Dictionary**: $O(V)$ auxiliary memory. Stores each predecessor pointer once. Path reconstructed in $O(d)$ time.
- **Queue Paths**: $O(d \cdot b^d)$ memory. Stores entire duplicated lists in each queue item. High memory overhead.